# Anomaly Detection on Pump Sensor Data
Unsupervised anomaly detection on multivariate time-series sensor data from an industrial pump.
Two approaches: Isolation Forest (classical) and LSTM Autoencoder (deep learning).
Dataset: Pump Sensor Data (Kaggle) — 52 sensors, ~220,000 timesteps.

In [1]:
import os
import json
from getpass import getpass

os.makedirs(os.path.expanduser("~/.config/kaggle"), exist_ok=True)

username = input("Kaggle username: ")
key = getpass("Kaggle API token: ")

kaggle_config = {"username": username, "key": key}

with open(os.path.expanduser("~/.config/kaggle/kaggle.json"), "w") as f:
    json.dump(kaggle_config, f)

os.chmod(os.path.expanduser("~/.config/kaggle/kaggle.json"), 0o600)
print("Kaggle credentials configured.")

Kaggle username: darrencameron
Kaggle API token: ··········
Kaggle credentials configured.


In [3]:
# Download data
os.makedirs("data/raw", exist_ok=True)
!pip install -q kaggle
!kaggle datasets download -d nphantawee/pump-sensor-data
!unzip -q pump-sensor-data.zip -d data/raw/
!ls data/raw/

Dataset URL: https://www.kaggle.com/datasets/nphantawee/pump-sensor-data
License(s): unknown
pump-sensor-data.zip: Skipping, found more recently modified local copy (use --force to force download)
sensor.csv


## Imports

In [4]:
import os
import json
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)

numpy: 2.0.2
pandas: 2.2.2
torch: 2.10.0+cpu


## Load Data and Exploration

In [5]:
df = pd.read_csv("data/raw/sensor.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

(220320, 55)
   Unnamed: 0            timestamp  sensor_00  sensor_01  sensor_02  \
0           0  2018-04-01 00:00:00   2.465394   47.09201    53.2118   
1           1  2018-04-01 00:01:00   2.465394   47.09201    53.2118   
2           2  2018-04-01 00:02:00   2.444734   47.35243    53.2118   
3           3  2018-04-01 00:03:00   2.460474   47.09201    53.1684   
4           4  2018-04-01 00:04:00   2.445718   47.13541    53.2118   

   sensor_03  sensor_04  sensor_05  sensor_06  sensor_07  ...  sensor_43  \
0  46.310760   634.3750   76.45975   13.41146   16.13136  ...   41.92708   
1  46.310760   634.3750   76.45975   13.41146   16.13136  ...   41.92708   
2  46.397570   638.8889   73.54598   13.32465   16.03733  ...   41.66666   
3  46.397568   628.1250   76.98898   13.31742   16.24711  ...   40.88541   
4  46.397568   636.4583   76.58897   13.35359   16.21094  ...   41.40625   

   sensor_44  sensor_45  sensor_46  sensor_47  sensor_48  sensor_49  \
0  39.641200   65.68287   50.925

In [6]:
print(df["machine_status"].value_counts())
print(f"\nTotal rows: {len(df)}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")

machine_status
NORMAL        205836
RECOVERING     14477
BROKEN             7
Name: count, dtype: int64

Total rows: 220320

Date range: 2018-04-01 00:00:00 to 2018-08-31 23:59:00
